In [37]:
import numpy as np
import matplotlib.pyplot as plt
import random
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torchsummary import summary
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import torchvision
from PIL import Image

train_data = np.load('Numarul1-train/1_1_train.npz', allow_pickle=True)
x_train, y_train = train_data['x'], train_data['y']
x_train = x_train.astype(np.float32)

test_data = np.load('Numarul1-test/1_1_test.npz', allow_pickle=True)
x_test, y_test = test_data['x'], test_data['y']
x_test = x_test.astype(np.float32)

N_CLASSES = len(np.unique(y_train))
print(f"Numarul de clase este: {N_CLASSES}")

X_train = torch.from_numpy(x_train)
X_test = torch.from_numpy(x_test)

print(X_train.shape, X_train.dtype)

mean = torch.mean(X_train, dim=0)
std = torch.std(X_train, dim=0)
X_train_norm = (X_train - mean) / std
X_test_norm = (X_test - mean) / std

n = X_train_norm.shape[0]
K = (X_train_norm.T @ X_train_norm) / (n - 1)
D, V = torch.linalg.eigh(K)

idx = torch.argsort(D, descending=True)
D = D[idx]
V = V[:, idx]

x_train = X_train_norm @ V[:, :400]
x_test = X_test_norm @ V[:, :400]

class CustomDataset(Dataset):
    def __init__(self, x, y):
        self.feature_vectors = x
        self.labels = y

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        feature_vector = self.feature_vectors[idx]
        label = self.labels[idx]

        feature_vector = torch.as_tensor(feature_vector, dtype=torch.float32)
        label = torch.as_tensor(label, dtype=torch.int64)

        return feature_vector, label

train_dataset = CustomDataset(x_train, y_train)
test_dataset = CustomDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


class Model(nn.Module):
    def __init__(self, in_features=400, hidden=128, n_classes=N_CLASSES, act=nn.functional.relu):
        super().__init__()
        self.act = act
        self.fc1 = nn.Linear(in_features=in_features, out_features=256)
        self.fc2 = nn.Linear(in_features=256, out_features=in_features)
        self.fc3 = nn.Linear(in_features=in_features, out_features=hidden)
        self.fc4 = nn.Linear(in_features=hidden, out_features=n_classes)

    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        x = self.act(self.fc3(x))
        x = self.fc4(x)

        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model(act=nn.functional.relu).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
loss_fn = nn.CrossEntropyLoss()
epochs = 10

def train(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(-1) == y).sum().item()
        total += y.size(0)
    return total_loss / len(loader), correct / total

def test(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out, y)
            total_loss += loss.item()
            correct += (out.argmax(-1) == y).sum().item()
            total += y.size(0)
        return total_loss / len(loader), correct / total

best = 0
for epoch in tqdm(range(epochs)):
    train_loss, train_acc = train(model, train_loader, optimizer, loss_fn, device)
    test_loss, test_acc = test(model, test_loader, loss_fn, device)
    print(f"Epoch {epoch}: train loss={train_loss:.4f}, acc={train_acc:.4f} | test loss={test_loss:.4f}, acc={test_acc:.4f}")
    if test_acc > best:
        best = test_acc
        torch.save(model, 'model.pt')

def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            correct += (out.argmax(-1) == y).sum().item()
            total += y.size(0)
        return correct / total


model = torch.load('model.pt', weights_only=False)
test_acc = evaluate(model, test_loader, device)
print(f'{test_acc:.4f}')


Numarul de clase este: 6
torch.Size([30000, 1024]) torch.float32


 10%|█         | 1/10 [00:00<00:07,  1.27it/s]

Epoch 0: train loss=1.3752, acc=0.4583 | test loss=1.2440, acc=0.5190


 20%|██        | 2/10 [00:01<00:05,  1.36it/s]

Epoch 1: train loss=1.1280, acc=0.5624 | test loss=1.1658, acc=0.5525


 30%|███       | 3/10 [00:02<00:05,  1.33it/s]

... (11 outputs left out)

 90%|█████████ | 9/10 [00:06<00:00,  1.40it/s]

Epoch 8: train loss=0.2564, acc=0.9088 | test loss=1.9529, acc=0.5562


100%|██████████| 10/10 [00:07<00:00,  1.38it/s]

Epoch 9: train loss=0.2097, acc=0.9250 | test loss=2.1799, acc=0.5452
0.5600
